# Agentic Legacy Integrator for Mainframe-to-ML Migration
## 1. Legacy Input Analysis & Preprocessing

**Observation:**
The input is a legacy mainframe JCL (Job Control Language) script used for DFSMS (Data Facility Storage Management Subsystem) configuration. It contains several steps (`STEP021` to `STEP065`) that define storage groups, data classes, and storage classes for a DB2 database.

**Cleaning Strategy:**
1.  **Noise Reduction:** Filter out empty lines, JCL comments starting with `//*`, and generic system DD statements (like `SYSPRINT`, `SYSUDUMP`).
2.  **Step Identification:** Use regex to identify executable steps (e.g., `//STEP... EXEC`).
3.  **Command Parsing:** Extract the core `ISPSTART CMD(...)` blocks where the actual business logic resides.
4.  **Rule Extraction:** Use regex to parse parameters (e.g., `HIGHTHRS(85)`) into a structured JSON format (Key-Value pairs).

**Ambiguities:**
* Many parameters have empty parentheses (e.g., `MIGSYSNM()`). We need to decide whether to map these to `null` or omit them in the modern schema.
* The connection between `SCDS('CPAC.DFSMS.SCDS')` and the defined classes implies a relational dependency that needs to be preserved.

In [1]:
import json
import re

def parse_jcl_to_json(file_path):
    """
    Reads a JCL file line-by-line and extracts business rules cleanly.
    """
    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()
    except FileNotFoundError:
        return f"Error: Could not find the file at {file_path}"

    structured_data = {
        "metadata": {
            "source_type": "Mainframe JCL",
            "description": "DFSMS Storage Management Rules"
        },
        "steps": []
    }

    current_step = None
    in_cmd = False
    cmd_buffer = ""

    for line in lines:
        # Step 1: Clean the line
        clean_line = line.strip()
        
        clean_line = re.sub(r'\[.*?\]', '', clean_line).strip()    

        if not clean_line or clean_line.startswith('//*'):
            continue

        # Step 2: Detect the Step Name (e.g., //STEP031 EXEC ...)
        step_match = re.match(r'^//(STEP\d+)\s+EXEC', clean_line)
        if step_match:
            current_step = step_match.group(1)
            continue

        # Step 3: Detect the start of a Command block
        if "CMD(" in clean_line:
            in_cmd = True
            cmd_buffer = clean_line.split("CMD(")[1]
            continue

        # Step 4: Accumulate parameters until the command closes
        if in_cmd:
            cmd_buffer += " " + clean_line
            if clean_line == ")" or (clean_line.endswith(")") and "+" not in clean_line):
                
                cmd_buffer = cmd_buffer.replace('+', ' ')
                
                action_match = re.search(r'(DEFINE|DISPLAY|ALTER)', cmd_buffer)
                action = action_match.group(1) if action_match else "UNKNOWN"
                
                params = re.findall(r'([A-Z0-9]+)\((.*?)\)', cmd_buffer)
                rules = {k: (v.strip() if v.strip() else None) for k, v in params}
                
                structured_data["steps"].append({
                    "step_name": current_step or "UNKNOWN_STEP",
                    "action": action,
                    "extracted_rules": rules
                })
                
                in_cmd = False
                cmd_buffer = ""

    return structured_data

# --- Execution ---
file_path = '../data/raw/sms'
parsed_rules = parse_jcl_to_json(file_path)

if isinstance(parsed_rules, str):
    print(parsed_rules)
else:
    with open('../data/processed/structured_rules.json', 'w') as out_file:
        json.dump(parsed_rules, out_file, indent=4)
        
    print("Parsing Successful! Here is a preview of the extracted rules:\n")
    print(json.dumps(parsed_rules, indent=4)[:800] + "\n...\n}")

Parsing Successful! Here is a preview of the extracted rules:

{
    "metadata": {
        "source_type": "Mainframe JCL",
        "description": "DFSMS Storage Management Rules"
    },
    "steps": [
        {
            "step_name": "STEP031",
            "action": "DEFINE",
            "extracted_rules": {
                "SCDS": "'CPAC.DFSMS.SCDS'",
                "STORGRP": "SGIXXX",
                "DESCR": "STORAGE GROUP FOR DB2",
                "AUTOMIG": "N",
                "MIGSYSNM": null,
                "AUTOBKUP": "N",
                "BKUPSYS": null,
                "AUTODUMP": null,
                "DMPSYSNM": null,
                "OVRFLOW": null,
                "EXTSGNM": null,
                "DUMPCLAS": null,
                "HIGHTHRS": "85",
                "LOWTHRS": "30",
                "GUARBKFR": "NOLIMIT",
             
...
}


## 2. Business Rule Extraction & Mapping

### 2.1. Core Business Rules Extracted
Based on the JSON payload generated in the previous step, several critical business rules and system constraints were identified from the legacy DFSMS configuration:

* **Threshold Logic (Auto-scaling Triggers):** The legacy system defines strict capacity thresholds. `HIGHTHRS: 85` indicates an upper limit of 85% utilization before action is required, and `LOWTHRS: 30` indicates a lower bound of 30%.
* **Resource Constraints:** `VOLCNT: 255` restricts the maximum number of storage volumes that can be allocated to this data class.
* **Availability & SLA Requirements:** `ACCSBTY: C` (Continuous) enforces a high-availability requirement, meaning the database (DB2) requires zero-downtime access.
* **Data Lifecycle & Retention:** `GUARBKFR: NOLIMIT` implies a continuous or unlimited backup guarantee without a strict expiration frequency.
* **Storage Formatting:** `DSNMTYP: EXT` and `IFEXT: R` define the structural requirements for the datasets (Extended format, Required).

### 2.2. Rule-to-Schema Mapping Table

| Legacy Concept (DFSMS) | Extracted Value | Modern Equivalent (Cloud / API) | Target JSON/YAML Field | Rationale |
| :--- | :--- | :--- | :--- | :--- |
| `STORGRP` (Storage Group) | `SGIXXX` | Cloud Storage Pool / K8s StorageClass | `storagePoolId` | Groups similar storage resources. Maps perfectly to a Kubernetes StorageClass. |
| `HIGHTHRS` / `LOWTHRS` | `85` / `30` | Auto-scaling Utilization Targets | `autoScaling.targetUtilization` | In modern cloud, static thresholds are replaced by dynamic auto-scaling policies based on percentage utilization. |
| `VOLCNT` | `255` | Max Resource Quota | `resourceQuotas.maxVolumes` | Hard limits in legacy systems translate to tenant or namespace resource quotas in modern platforms. |
| `ACCSBTY` | `C` (Continuous) | High Availability (Multi-AZ) | `sla.highAvailability: true` | Continuous access in mainframe translates to Multi-AZ (Availability Zone) deployment in AWS/Azure. |
| `AUTOMIG` | `N` (No) | Auto-Tiering Policy | `lifecyclePolicy.autoTiering: false` | Legacy manual migration indicates that dynamic storage tiering (e.g., S3 Standard to Glacier) should be disabled. |

### 2.3. Assumptions and Ambiguities
* **Assumption 1:** The legacy parameter `ACCSBTY: C` is assumed to require a Multi-Region or Multi-AZ architecture in the target cloud environment to guarantee continuous access.
* **Assumption 2:** Many parameters (e.g., `MIGSYSNM`, `AUTODUMP`) were extracted as `null`. We assume these were either default behaviors or non-essential features in the legacy system. In the modern schema, we will omit them to maintain a clean API contract.
* **Ambiguity:** The exact metric for `HIGHTHRS: 85` (whether it's IOPS, throughput, or purely disk capacity) is implied to be capacity based on typical DFSMS setups, but would require business stakeholder validation in a real-world scenario.

In [3]:
import json

def map_legacy_to_modern_schema(legacy_json):
    """
    Transforms the extracted legacy rules into a modern REST API payload
    suitable for a Cloud Storage Provisioning Microservice.
    """
    
    # We will extract specific rules from the legacy JSON to build our modern schema
    storage_group_rules = {}
    data_class_rules = {}
    storage_class_rules = {}

    for step in legacy_json.get("steps", []):
        if step.get("action") == "DISPLAY":
            continue
        rules = step.get("extracted_rules", {})
        if "STORGRP" in rules:
            storage_group_rules = rules
        if "DCNAME" in rules:
            data_class_rules = rules
        if "STCNAME" in rules:
            storage_class_rules = rules

    # Building the modern JSON schema (API Payload)
    modern_payload = {
        "apiVersion": "v1",
        "kind": "StorageProvisioningRequest",
        "metadata": {
            "serviceName": "DB2_Modernization_DB",
            "environment": "production"
        },
        "spec": {
            "storageClass": {
                "name": storage_class_rules.get("STCNAME", "default-class"),
                "description": storage_class_rules.get("DESCR", ""),
                "sla": {
                    # 'C' means Continuous, which we map to High Availability
                    "highAvailability": True if storage_class_rules.get("ACCSBTY") == "C" else False
                }
            },
            "capacityPlanning": {
                "quotas": {
                    "maxVolumes": int(data_class_rules.get("VOLCNT", 100))
                },
                "autoScaling": {
                    "enabled": True,
                    "scaleOutThresholdPercent": int(storage_group_rules.get("HIGHTHRS", 80)),
                    "scaleInThresholdPercent": int(storage_group_rules.get("LOWTHRS", 20))
                }
            },
            "dataLifecycle": {
                "autoTieringEnabled": True if storage_group_rules.get("AUTOMIG") == "Y" else False,
                "backupPolicy": storage_group_rules.get("GUARBKFR", "STANDARD")
            }
        }
    }
    
    return modern_payload

# --- Execution ---
# Load the processed legacy rules
with open('../data/processed/structured_rules.json', 'r') as f:
    legacy_data = json.load(f)

# Generate the modern schema
modernized_schema = map_legacy_to_modern_schema(legacy_data)

# Save the modern schema
with open('../data/processed/modern_api_payload.json', 'w') as f:
    json.dump(modernized_schema, f, indent=4)

print("Modernization Mapping Complete! Here is the target API schema:\n")
print(json.dumps(modernized_schema, indent=4))

Modernization Mapping Complete! Here is the target API schema:

{
    "apiVersion": "v1",
    "kind": "StorageProvisioningRequest",
    "metadata": {
        "serviceName": "DB2_Modernization_DB",
        "environment": "production"
    },
    "spec": {
        "storageClass": {
            "name": "SCIXXX",
            "description": "STORAGE CLASS FOR DB2",
            "sla": {
                "highAvailability": true
            }
        },
        "capacityPlanning": {
            "quotas": {
                "maxVolumes": 255
            },
            "autoScaling": {
                "enabled": true,
                "scaleOutThresholdPercent": 85,
                "scaleInThresholdPercent": 30
            }
        },
        "dataLifecycle": {
            "autoTieringEnabled": false,
            "backupPolicy": "NOLIMIT"
        }
    }
}


## 3. Agent Workflow Development & Evaluation

### 3.1. Workflow Design (Multi-Stage Pipeline)
To ensure high-quality translation of legacy logic, we designed a multi-stage agentic pipeline using a local LLM (`Mistral-7B-Instruct-v0.2`). Running the model locally ensures data privacy for sensitive enterprise logic.

The pipeline consists of four stages:
1.  **Parsing/Retrieval:** Loading the cleaned intermediate JSON (`structured_rules.json`).
2.  **Reasoning & Extraction:** Prompting the LLM to act as an analyst to identify business rules, entities, and migration risks without worrying about formatting.
3.  **Structured Generation:** Prompting the LLM specifically to generate a strict JSON target schema based on the legacy input.
4.  **Validation:** Using Python to programmatically verify that the LLM's output is valid JSON and contains the required modern fields.

### 3.2. Prompt Structure (Mistral-Specific)
We utilized Mistral's native instruction format `<s>[INST] {prompt} [/INST]` to clearly separate instructions from data. We split the tasks into two distinct prompts to reduce cognitive load on the 7B model and prevent "hallucinations" (making up data).

### 3.3. Potential Challenges & Mitigations
* **Ambiguous Business Logic:** Legacy parameters like `ACCSBTY: C` are highly contextual. We instructed the LLM to highlight these as "Migration Risks" rather than blindly guessing.
* **Poor Source Formatting:** Addressed in Step 1 by converting raw JCL to a structured intermediate JSON.
* **Hallucinated JSON Formatting:** 7B parameter models sometimes add conversational text (e.g., "Here is your JSON..."). We mitigated this by strictly instructing the model to output *only* JSON and implemented a validation parsing step in Python.

In [6]:
import json
from llama_cpp import Llama

# STAGE 0: Setup and Initialization
print("Loading Mistral-7B Model... (This might take a few seconds)")

llm = Llama(
    model_path="../models/mistral-7b-instruct-v0.2.Q4_K_M.gguf", 
    n_ctx=4096, 
    verbose=False 
)

# Load the intermediate data from Step 1
with open('../data/processed/structured_rules.json', 'r') as f:
    legacy_data = json.dumps(json.load(f), indent=2)

# STAGE 1: Reasoning & Extraction
def extract_business_logic(data):
    print("Stage 1: Extracting Business Logic & Risks...")
    prompt = f"""[INST] You are an expert Mainframe-to-Cloud Migration Architect. 
Analyze the following legacy DFSMS JSON data and provide a concise report containing:
1. Core Business Rules (e.g., capacity limits, thresholds like HIGHTHRS or VOLCNT).
2. Migration Risks and Ambiguities.
3. Proposed Modernization Steps.

CRITICAL INSTRUCTION: Keep your response brief.

Legacy Data:
{data}
[/INST]"""

    response = llm(prompt, max_tokens=1500, temperature=0.3)
    return response['choices'][0]['text'].strip()

# STAGE 2: Structured Generation
def generate_modern_schema(data):
    print("Stage 2: Generating Target JSON Schema...")
    prompt = f"""[INST] You are an expert API Designer. Convert the following legacy system data into a modern REST API JSON payload (e.g., for Cloud Storage Provisioning).
Map keys like 'VOLCNT' to 'maxVolumes', 'HIGHTHRS' to 'scaleOutThreshold', etc.

CRITICAL INSTRUCTIONS:
1. Output ONLY valid JSON. 
2. OMIT ALL NULL VALUES. If a field in the legacy data is null, DO NOT include it in the output.
3. Do not include markdown code blocks (like ```json), and do not include any introductory text.

Legacy Data:
{data}
[/INST]"""

    response = llm(prompt, max_tokens=1500, temperature=0.1) 
    return response['choices'][0]['text'].strip()

# STAGE 3: Validation
def validate_schema(schema_text):
    print("Stage 3: Validating Output...")
    try:
        parsed_json = json.loads(schema_text)
        print("Validation Passed: Output is valid JSON.")
        return parsed_json
    except json.JSONDecodeError as e:
        print(f"Validation Failed: LLM did not return pure JSON. Error: {e}")
        print("Raw LLM Output Snippet:\n", schema_text[-300:])
        return None

# EXECUTE PIPELINE
print("-" * 50)
report = extract_business_logic(legacy_data)
print("\n--- MIGRATION REPORT ---\n", report)
print("-" * 50)

raw_schema = generate_modern_schema(legacy_data)
final_schema = validate_schema(raw_schema)

if final_schema:
    # Save the successful output
    with open('../data/processed/modern_api_contract.json', 'w') as f:
        json.dump(final_schema, f, indent=4)
    print("\n--- MODERN API SCHEMA ---\n", json.dumps(final_schema, indent=4))

Loading Mistral-7B Model... (This might take a few seconds)


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


--------------------------------------------------
Stage 1: Extracting Business Logic & Risks...

--- MIGRATION REPORT ---
 Report:

1. Core Business Rules:
- Capacity limits: HIGHTHRS = 85, LOWTHRS = 30, VOLCNT = 255.
- Thresholds: HIGHTHRS and LOWTHRS for storage management.
- Storage Classes: DEFINED for DB2 with no automation.
- Data Classes: DEFINED for DB2 with REUSE = null and RECACCB = "U".
- Maximum volume: MAXVOL = null.

2. Migration Risks and Ambiguities:
- Unknown action in STEP041, which might cause unexpected behavior during migration.
- Missing information in some rules such as DRTBIAS, SEQMSRES, SEQBIAS, INIARESS, SUSDTRT, AVALBTY, ACCSBTY, GURNTSPC, GUASYNWR, MULTITSG, CFCACSTN, CFDTWGHT, CFSEQWHT, SPCAVREC, SPCAVVAL, SPCSEC, SPCDIR, REXPPDT, COMPTN, MDTYPE, RECTECH, DSNMTYP, IFEXT, EXTADDRS, and LOG.
- The meaning and impact of some abbreviations like BATSCRW, BATSCRD, BREDIMAX, and BDISPMAX are unclear.

3. Proposed Modernization Steps:
- Thoroughly investigate the 

## Workflow Evaluation & Reflection

Based on the execution of the Mistral-7B agentic pipeline, here is the evaluation of the generated output against the project criteria:

### 1. Consistency and Reasoning (Strengths)
* **Accurate Rule Extraction:** The agent successfully parsed the intermediate JSON and accurately extracted critical capacity limits (`HIGHTHRS = 85`, `VOLCNT = 255`).
* **Intelligent Risk Assessment:** Instead of blindly translating, the model demonstrated strong analytical capabilities. It correctly flagged the `UNKNOWN` action in `STEP041` as a migration risk and identified legacy abbreviations (`BATSCRW`, `BATSCRD`) as ambiguous logic requiring human-in-the-loop (HITL) review.
* **Modern Schema Mapping:** It successfully mapped legacy mainframe concepts to modern cloud-native parameters (e.g., mapping `HIGHTHRS` to `scaleOutThreshold` and `VOLCNT` to `maxVolumes`).

### 2. Formatting Quality & System Clarity
* **Validation Success:** The output passed the strict Python validation stage, generating a perfectly formatted, machine-readable JSON API contract.
* **Pipeline Architecture:** The decision to split the workflow into discrete "Reasoning" and "Generation" stages prevented the model from mixing conversational text with the JSON payload.

### 3. Challenges & LLM Quirks (Hallucinations/Limitations)
* **Negative Prompting Failure:** Despite critical instructions to "OMIT ALL NULL VALUES," the 7B parameter model struggled to maintain this negative constraint towards the end of its context window, outputting several `null` fields inside the `dataClass` block (e.g., `"reusePolicy": null`). 
* **Entity Resolution:** The model hallucinated the entity names slightly, using the overarching SCDS name (`CPAC.DFSMS.SCDS`) for all classes instead of their specific legacy IDs (`SCIXXX`, `DCIXXX`).

### 4. Proposed Enhancements (Next Steps)
To improve this baseline workflow, relying entirely on the LLM for strict formatting constraints should be minimized. In the next iteration, a programmatic Python filter should be added to the `validate_schema` function to recursively strip `null` values and correct entity IDs, leaving the LLM to focus purely on semantic translation.